# L8: ML实战项目


# L8: ML实战项目

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 独立完成从数据探索到模型部署的完整ML Pipeline
2. 掌握特征工程的常用方法（编码、标准化、特征组合）
3. 使用交叉验证进行可靠的模型评估
4. 使用GridSearchCV进行超参数调优
5. 完成一个Kaggle入门级竞赛并提交结果

## 2. 核心知识点

### 2.1 特征工程

**数值型特征处理**:
- 标准化 (StandardScaler): z = (x - μ) / σ
- 归一化 (MinMaxScaler): z = (x - x_min) / (x_max - x_min)

**类别型特征处理**:
- 标签编码 (LabelEncoder): 适用于有序类别
- 独热编码 (OneHotEncoder): 适用于无序类别


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pandas as pd

df = pd.DataFrame({
    '年龄': [25, 30, np.nan, 40],
    '城市': ['北京', '上海', '北京', '深圳'],
    '收入': [5000, 8000, 6000, 12000],
    '购买': [1, 0, 1, 0]
})

numeric_features = ['年龄', '收入']
categorical_features = ['城市']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='未知')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X = df.drop('购买', axis=1)
y = df['购买']
X_processed = preprocessor.fit_transform(X)
print(f"预处理后特征形状: {X_processed.shape}")


### 2.2 交叉验证

**K折交叉验证**: 将数据分为K份，轮流用K-1份训练、1份验证


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

cv_scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
print(f"F1分数: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# GridSearchCV自动调参
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(X, y)
print(f"最优参数: {grid_search.best_params_}")


### 2.3 Kaggle入门竞赛实战（Titanic）


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 1. 加载数据
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 特征工程
def feature_engineering(df):
    df = df.copy()
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['FareBin'] = pd.qcut(df['Fare'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['AgeBin'] = pd.cut(df['Age'], 5, labels=['C1', 'C2', 'C3', 'C4', 'C5'])
    return df

train = feature_engineering(train)
test = feature_engineering(test)

# 3. 特征编码
from sklearn.preprocessing import LabelEncoder
for col in ['Sex', 'Title', 'FareBin', 'AgeBin', 'Embarked']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# 4. 选择特征并训练
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Title', 'FamilySize', 'IsAlone']
X = train[features].fillna(0)
y = train['Survived']
X_test = test[features].fillna(0)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f"交叉验证准确率: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

rf.fit(X, y)
predictions = rf.predict(X_test)
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)
print("提交文件已生成: submission.csv")


## 3. 代码示例


In [ ]:
"""
L8-ml-pipeline.py
完整ML Pipeline演示：从数据加载到模型调优到结果提交
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("=" * 60)
print("ML完整Pipeline：从数据到模型到预测")
print("=" * 60)

from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "逻辑回归": LogisticRegression(max_iter=1000, random_state=42),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
    "梯度提升": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    results[name] = {"train": train_acc, "test": test_acc, "cv_mean": cv_scores.mean()}
    print(f"{name}: 训练={train_acc:.3f}, 测试={test_acc:.3f}, CV={cv_scores.mean():.3f}")

# 超参数调优
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f"最优参数: {grid_search.best_params_}")
print(f"最优CV分数: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print(f"测试集准确率: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=data.target_names))

# 可视化混淆矩阵和特征重要性
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(data.target_names)
axes[0].set_yticklabels(data.target_names)
axes[0].set_xlabel('预测')
axes[0].set_ylabel('真实')
axes[0].set_title('混淆矩阵')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20)

feature_importance = pd.DataFrame({
    'feature': data.feature_names,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

axes[1].barh(feature_importance['feature'], feature_importance['importance'], color='#3498db')
axes[1].set_xlabel('重要性')
axes[1].set_title('Top 10 特征重要性')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("l8_final_model.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. 练习题

1. **Pipeline构建**: 使用Pipeline和ColumnTransformer构建完整数据处理+模型训练Pipeline。
2. **特征工程实战**: 对UCI Adult收入数据集进行完整特征工程处理。
3. **交叉验证对比**: 比较5折、10折、留一法三种CV方法的差异与时间成本。
4. **超参数调优**: 使用RandomizedSearchCV对XGBoost进行调优，比较与GridSearchCV的效率。
5. **Kaggle入门**: 完成Kaggle入门竞赛，提交分析报告。

## 5. 延伸阅读

- **网站**: Kaggle入门教程 — https://www.kaggle.com/docs/getting-started
- **书籍**: Aurélien Géron, *Hands-On Machine Learning*（第3版）
- **工具**: MLflow — 开源ML生命周期管理平台 https://mlflow.org/
- **GitHub**: A Collection of Data Science Interview Questions — https://github.com/ShuaiW/data-science-interview-questions

---

# 附录：课程进度检查清单

## L5 检查点
- [ ] 能区分四种机器学习范式
- [ ] 理解偏差-方差权衡的本质
- [ ] 能解释过拟合的原因及缓解策略

## L6 检查点
- [ ] 能推导线性回归与逻辑回归的损失函数
- [ ] 理解SVM间隔最大化的原理
- [ ] 能计算并解释各种评估指标

## L7 检查点
- [ ] 能解释K-Means的EM算法流程
- [ ] 理解PCA降维的数学原理
- [ ] 掌握异常检测的主要方法

## L8 检查点
- [ ] 能独立完成完整的ML Pipeline
- [ ] 掌握特征工程的主要方法
- [ ] 能使用GridSearchCV进行超参数调优

---

*文档版本：v1.0 | 适用教材：AI 人工智能基础教程 Week 3-4*
